# Plant Leaf Disease Detection using Fine-Tuned ResNet-18

This practical fine-tunes a pretrained ResNet-18 model to classify bean leaf images into three categories: **healthy**, **angular leaf spot**, and **bean rust**.

Dataset: Hugging Face `beans` dataset. It downloads automatically. No external SQL database or Kaggle account is required.

Workflow: load dataset → preprocess images → load pretrained model → freeze/unfreeze layers → configure Trainer → fine-tune → evaluate → save → reload → predict.

## 1. Install required libraries

In [ ]:
!pip install -U datasets transformers torch accelerate scikit-learn matplotlib pillow

## 2. Import libraries

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
from PIL import Image

## 3. Load the dataset

In [ ]:
dataset = load_dataset("beans")
print(dataset)

class_names = dataset["train"].features["labels"].names
print("Classes:", class_names)
print("Training:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Testing:", len(dataset["test"]))

## 4. Select manageable samples

In [ ]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(min(800, len(dataset["train"]))))
validation_dataset = dataset["validation"].shuffle(seed=42).select(range(min(150, len(dataset["validation"]))))
test_dataset = dataset["test"].shuffle(seed=42).select(range(min(150, len(dataset["test"]))))

print(len(train_dataset), len(validation_dataset), len(test_dataset))

## 5. Display sample images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    sample = train_dataset[i]
    ax.imshow(sample["image"])
    ax.set_title(class_names[sample["labels"]])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Load pretrained image processor

In [ ]:
model_checkpoint = "microsoft/resnet-18"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint)

## 7. Preprocess images

In [ ]:
def preprocess_images(examples):
    images = [image.convert("RGB") for image in examples["image"]]
    inputs = image_processor(images=images, return_tensors="np")
    inputs["labels"] = examples["labels"]
    return inputs

train_dataset = train_dataset.map(preprocess_images, batched=True, remove_columns=train_dataset.column_names)
validation_dataset = validation_dataset.map(preprocess_images, batched=True, remove_columns=validation_dataset.column_names)
test_dataset = test_dataset.map(preprocess_images, batched=True, remove_columns=test_dataset.column_names)

train_dataset.set_format(type="torch")
validation_dataset.set_format(type="torch")
test_dataset.set_format(type="torch")
print(train_dataset[0].keys())

## 8. Load pretrained ResNet-18 model

In [ ]:
id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in enumerate(class_names)}

model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(class_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

## 9. Freeze and unfreeze layers

In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model.classifier.parameters():
    parameter.requires_grad = True

for name, parameter in model.named_parameters():
    if name.startswith("resnet.encoder.stages.3"):
        parameter.requires_grad = True

print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters:", sum(p.numel() for p in model.parameters()))

## 10. Define evaluation metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## 11. Set training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./plant-disease-resnet-results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

## 12. Create Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics
)

## 13. Fine-tune the model

In [ ]:
trainer.train()

## 14. Evaluate on the test dataset

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

## 15. Display confusion matrix

In [ ]:
prediction_output = trainer.predict(test_dataset)
predicted_labels = np.argmax(prediction_output.predictions, axis=1)
true_labels = prediction_output.label_ids

cm = confusion_matrix(true_labels, predicted_labels, labels=list(range(len(class_names))))
matrix_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
matrix_display.plot(xticks_rotation="vertical")
plt.title("Plant Leaf Disease Confusion Matrix")
plt.tight_layout()
plt.show()

## 16. Save the fine-tuned model and image processor

In [ ]:
save_directory = "./plant-disease-resnet"
trainer.save_model(save_directory)
image_processor.save_pretrained(save_directory)
print("Model saved to:", save_directory)

## 17. Reload the saved model

In [ ]:
image_processor = AutoImageProcessor.from_pretrained(save_directory)
model = AutoModelForImageClassification.from_pretrained(save_directory)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print("Model loaded on:", device)

## 18. Create prediction function

In [ ]:
def predict_leaf_disease(image):
    if not isinstance(image, Image.Image):
        image = Image.open(image)

    image = image.convert("RGB")
    inputs = image_processor(images=image, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(probabilities, dim=-1).item()
        confidence = probabilities[0, predicted_class].item()

    predicted_name = id2label[predicted_class]
    print("Prediction:", predicted_name)
    print("Confidence:", f"{confidence * 100:.2f}%")

    plt.imshow(image)
    plt.title(f"{predicted_name} ({confidence * 100:.2f}%)")
    plt.axis("off")
    plt.show()
    return predicted_name, confidence

## 19. Validate using a known test image

In [ ]:
sample = dataset["test"][0]
print("Actual class:", class_names[sample["labels"]])
predict_leaf_disease(sample["image"])

## 20. Upload your own leaf image

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    image_path = next(iter(uploaded))
    predict_leaf_disease(image_path)
except ImportError:
    image_path = input("Enter the image file path: ").strip()
    predict_leaf_disease(image_path)

## Notes

- This practical uses the Hugging Face **Beans** dataset, with three bean-leaf classes. It is not a general detector for every plant species.
- The dataset downloads automatically using `load_dataset("beans")`; no separate database installation is needed.
- The notebook uses a limited sample for a manageable first run. Increase the sample counts after confirming that the workflow runs.
- A GPU is recommended; CPU training may be slow.
- Validate using test metrics, the confusion matrix, and known test images.
- For your own image, use a clear, well-lit bean leaf photo. Model confidence is a score, not a guaranteed agricultural diagnosis.